# BONUS · B7 · Lakeflow Jobs — Triggers, Dependencies & Orchestration

> *This module is optional. You can work through it independently after the main course sessions.*

> *"The RetailHub pipeline works — now automate it. Configure triggers, define task dependencies, handle failures with repair runs, and monitor via system tables."*

## Learning Objectives

After completing this module you will be able to:

- **Create** a Lakeflow Job with multiple notebook tasks and dependencies
- **Configure** a Table Update trigger to chain two jobs automatically
- **Understand** the difference between job cluster and all-purpose cluster
- **Read** a Databricks Asset Bundle YAML job definition

## Workshop Overview

| Step | Focus |
|------|-------|
| **1–5** | Create `customer_pipeline` job with two tasks |
| **6–8** | Create `orders_pipeline` with Table Update trigger |
| **Reference** | YAML definitions for both jobs (Asset Bundle format) |

## Setup

In [0]:
%run ../../setup/00_setup

## Section 1: Workshop — Creating & Running Jobs in Databricks UI

In this section you will create two Lakeflow Jobs through the Databricks UI:

| Job | Tasks | Purpose |
|-----|-------|---------|
| **customer_pipeline** | `bronze_customer` → `silver_customer` | Ingest and cleanse customer data |
| **orders_pipeline** | `bronze_orders` → `silver_orders` → `gold_daily_orders` → `gold_summary` | Full orders medallion pipeline, triggered by customer updates |

> **Goal:** Create both jobs, configure task dependencies, set up a **Table Update trigger** so the orders pipeline runs automatically when `silver_customers` is updated, then execute the pipeline end-to-end.

### Step 1: Create a New Job — `customer_pipeline`

Navigate to **Jobs & Pipelines** in the left sidebar, then click **Create Job**.

![Create Job](../../../assets/images/training_2026/day3/6f524625f8464c29a2572b0a324333a5.webp)

### Step 2: Set the Job Name

Enter a **unique name** for your job (e.g., `<your_name>_customer_pipeline`).

![Job Name](../../../assets/images/training_2026/day3/ae8ab8cafdcf4690b5d9665cbf058798.webp)

> **Note:** The job name does not need to be globally unique — each job is identified by a unique **Job ID** assigned automatically by Databricks.

![Job Details — ID & Creator](../../../assets/images/training_2026/day3/460925af2b214f93a2252aed0ea86936.webp)

### Step 3: Configure Job Parameters

Add the following **job parameters** so that all tasks share the same catalog and source path:

| Parameter | Value |
|-----------|-------|
| `catalog` | `retailhub_<your_name>` |
| `source_path` | `/Volumes/retailhub_<your_name>/default/datasets` |

![Job Parameters](../../../assets/images/training_2026/day3/e24f8fbe50f642e0a28054972140dcb1.webp)

### Step 4: Add the First Task — `bronze_customer`

Click **Add task** and configure:

| Field | Value |
|-------|-------|
| **Task name** | `bronze_customer` |
| **Type** | Notebook |
| **Source** | Workspace |
| **Path** | Path to your `bronze_customers` notebook |
| **Compute** | Shared training cluster *(for workshop only — in production, use a dedicated Job cluster)* |

![Add Task](../../../assets/images/training_2026/day3/8d2245e891b849e5966aae38a0e33c5f.webp)

![Task Configuration](../../../assets/images/training_2026/day3/c89c69190d40463faae2b5186c78c271.webp)

Click **Create task** when done.

### Step 5: Add the Second Task — `silver_customer`

Repeat the same process for `silver_customer`:
- Set **Depends on** → `bronze_customer` (so it runs only after bronze succeeds)
- Point the notebook path to your `silver_customers` notebook

Your completed job should look like this:

![Customer Pipeline — 2 Tasks](../../../assets/images/training_2026/day3/8bdeb6d46c284e3c85bf1c04f4b27657.webp)

### Step 6: Create the Second Job — `orders_pipeline`

Create a **new job** following the same steps as above. Name it `<your_name>_orders_pipeline` and add all four medallion tasks with proper dependencies.

Your completed pipeline should look like this:

![Orders Pipeline — 4 Tasks](../../../assets/images/training_2026/day3/119b8804bea74938aff23d816140bf4b.webp)

### Step 7: Configure a Table Update Trigger

Navigate to the **Triggers** tab of the `orders_pipeline` job and add a **Table Update** trigger on `silver_customers`.

This means the orders pipeline will **start automatically** whenever the `silver_customers` table is updated by the customer pipeline.

![Table Update Trigger Configuration](../../../assets/images/training_2026/day3/d4dd6298fc9b41e9bc784878ccd8dc3a.webp)

### Step 8: Run the Pipeline

Click **Run now** on the `customer_pipeline` job first.

![Run Job](../../../assets/images/training_2026/day3/73685979cbb243b3af4e01fdcadb1a32.webp)

> **Expected result:** The customer pipeline completes successfully, which updates `silver_customers`. This triggers the orders pipeline automatically — both jobs should show a successful run in the **Run History** tab.

### Reference: YAML Definition — `orders_pipeline`

> The YAML below shows the **Databricks Asset Bundle** definition for the orders pipeline. This is the programmatic equivalent of what you configured in the UI above.

```yaml
resources:
  jobs:
    demo_orders_pipeline:
      name: demo_orders_pipeline
      trigger:
        pause_status: UNPAUSED
        table_update:
          table_names:
            - retailhub_trainer.silver.silver_customers
      tasks:
        - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/.../bronze_orders
            source: WORKSPACE
          existing_cluster_id: <CLUSTER_ID>
        - task_key: silver_orders
          depends_on:
            - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/.../silver_orders
            source: WORKSPACE
          existing_cluster_id: <CLUSTER_ID>
        - task_key: gold_daily_orders
          depends_on:
            - task_key: silver_orders
          notebook_task:
            notebook_path: /Workspace/.../gold_daily_orders
            source: WORKSPACE
          existing_cluster_id: <CLUSTER_ID>
        - task_key: gold_customer_orders_summary
          depends_on:
            - task_key: gold_daily_orders
          notebook_task:
            notebook_path: /Workspace/.../gold_customer_orders_summary
            source: WORKSPACE
          existing_cluster_id: <CLUSTER_ID>
      queue:
        enabled: true
      parameters:
        - name: catalog
          default: retailhub_trainer
        - name: source_path
          default: /Volumes/retailhub_trainer/default/datasets
```

### Reference: YAML Definition — `customer_pipeline`

```yaml
resources:
  jobs:
    demo_customer_pipeline:
      name: demo_customer_pipeline
      tasks:
        - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/.../bronze_customers
            source: WORKSPACE
          existing_cluster_id: <CLUSTER_ID>
        - task_key: silver_customer
          depends_on:
            - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/.../silver_customers
            source: WORKSPACE
          existing_cluster_id: <CLUSTER_ID>
      queue:
        enabled: true
      parameters:
        - name: catalog
          default: retailhub_trainer
        - name: source_path
          default: /Volumes/retailhub_trainer/default/datasets
```

> **Key differences:** The customer pipeline has **no trigger** (runs manually or on-demand), while the orders pipeline uses a **table_update trigger** on `silver_customers`.

## Summary

| Concept | Key Takeaway |
|---------|-------------|
| **Lakeflow Job** | Orchestrates notebook tasks with dependencies, triggers, and compute settings |
| **Task dependencies** | Define execution order; downstream tasks run only when upstream tasks succeed |
| **Table Update trigger** | Starts a job automatically when a specified Delta table is updated |
| **Job cluster** | Cost-optimized compute that terminates after the job — use for automated workloads |
| **All-purpose cluster** | Long-running shared compute — use for interactive / development work |
| **Asset Bundle YAML** | Declarative job definition enabling version control and CI/CD deployment |
| **Repair Run** | Re-executes only the failed task and its downstream dependents — saves compute cost |